# **Construye aplicaciones de IA más inteligentes: potencia los LLM con LangChain**

## Descripción general

En este laboratorio, adquirirás experiencia práctica utilizando LangChain para simplificar los procesos complejos necesarios para integrar capacidades avanzadas de IA en aplicaciones reales. Aplicarás las funcionalidades principales del framework LangChain y aprovecharás sus características innovadoras para construir aplicaciones más inteligentes, adaptativas y eficientes.

## Objetivos

Al completar este laboratorio, serás capaz de:

* Utilizar las funcionalidades principales del framework LangChain, incluyendo plantillas de prompts, cadenas (chains) y agentes, para mejorar la personalización de los LLM y la relevancia de sus respuestas.

* Explorar el enfoque modular de LangChain, que permite realizar ajustes dinámicos en prompts y modelos sin necesidad de realizar cambios extensos en el código.

* Mejorar aplicaciones basadas en LLM mediante la integración de técnicas de **generación aumentada por recuperación (RAG)** con LangChain. Aprenderás cómo esta integración permite una mayor precisión y respuestas más conscientes del contexto.


## Setup
### Importar las librerias

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from llm_config import get_llm

c:\Users\ferna\anaconda3\envs\rag_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

## Configurar los LLM

En esta seccion configuramnos los LLM usando nuestro entorno hibrido **DeepSeek + Ollama** utilizando el modulo `llm_config`:

- `lmm` :  Especifica el modulo a utilizar. `sdk` para deepseek modelo `deepseek-chat` y `oll` para LLM local Ollama modelo `qwen:4b`. Por defecto utiliza DeepSeek. 
- `params` :  Configura los parametros del modelo establecidos por defecto como: 
    - `temperature`: 1, Aleatoriedad del modelo.
    - `max_tokens`: 50, Longitud de la respuesta en DeepSeek
    - `top_p`: 1, Diversidad de palabras considerando la probabilidad acumulada (no usar junto con temperature).
    - `top_k`: 40, Diversidad de palabras considerando la frecuencia 
    - `frequency_penalty`: 0, Penalizacion de frecuencia - DeepSeek
    - `presence_penalty`: 0, Penalizacion de presencia - DeepSeek
    - `repeat_penalty`: 1.1 Penalizacion de repeticion - Ollama

Ejemplo de llamada con `llmconfig`:
```python
get_llm(llm="dsk", params={"temperature": 0.8, "max_tokens": 30})

## Estructura de la respuesta de los LLM

La respuesta no siempre es un simple texto. En muchos casos (especialmente con Chat Models como DeepSeek), obtienes un objeto estructurado con mucha más información que pueden ser extraidas como objetos generados por el modelo que estes usando. 

EL objeto `content`, es el que tiene la respuesta del prompt, para que se muestre solo ese usamos `response.content`. Este viene formateado en Markdown. Para vizualizar correctamente el formato importamos librerias de vizualizacion de jupyter: 

In [3]:
from IPython.display import display, Markdown

Definimos una funcion para reemplazar a `print()` en la respuesta que muestre el texto en el formato correcto: 

In [4]:
def show(response):
    content = response.content if hasattr(response, "content") else response
    display(Markdown(content))

Ahora solamante llamamos a show(response) y devolvera `content` vizualizado correctamnente. 

### Recapitulando: 

En el Lab anterior se abordo los diferentes tipos de prompt, el metodo LCEL para crear `chains` y `templates` que generen interacciones reurtilizables con la IA, en el siguiente ejemplo vemos los componentes y temas abordados:

In [ ]:
content = """
        El marketing digital es el conjunto de estrategias y técnicas que se utilizan para promocionar productos o servicios a través de canales digitales. Incluye herramientas como redes sociales, email marketing, SEO (optimización para motores de búsqueda) y publicidad online. El SEO se centra en mejorar la visibilidad de una web en los resultados orgánicos de buscadores como Google, mientras que la publicidad online permite llegar a audiencias específicas mediante anuncios pagados.
"""

question = "¿Qué técnica del marketing digital se centra en mejorar la visibilidad en buscadores?"

# Template
template = """
Responde a la siguiente {question} basándote en {content} proporcionado:
Si no estás seguro de la respuesta, responde: "No estoy seguro de la respuesta".

Respuesta:
"""

# Promt
prompt = PromptTemplate.from_template(template)

# llm
llm = get_llm(llm="dsk", params={"temperature": 0.5})

# parser
parser = StrOutputParser()

# Chain
qa_chain = (
    prompt
    | llm
    | parser
)

response = qa_chain.invoke({"question": question, "content": content})
show(response)

## **Modelo de chat**

Los modelos de chat permiten asignar roles distintos a los mensajes de una conversación, lo que ayuda a diferenciar entre mensajes de la IA, del usuario y de instrucciones como los mensajes de sistema.
